# Pràctica 4: Embeddings distribucionals i contextuals

## Introducció

L'objectiu principal d'aquesta pràctica és entrenar i avaluar models d'**embeddings distribucionals i contextuals** per a tasques de similitud semàntica en espanyol.

Els **embeddings distribucionals** (com Word2Vec o GloVe) representen paraules com a vectors densos basant-se en la hipòtesi distribucional: paraules que apareixen en contextos similars tendeixen a tenir significats similars. Aquests vectors es calculen a partir d'un corpus gran i són estàtics, és a dir, cada paraula té sempre el mateix vector independentment del context.

Els **embeddings contextuals** (com BERT o RoBERTa) superen aquesta limitació generant representacions dinàmiques que canvien segons el context de la frase. Això permet capturar fenòmens com la polisèmia.

## Datasets

- **Multi-SimLex (Spanish):** conjunt de parells de paraules amb puntuacions de similitud semàntica anotades per humans. S'utilitza per a l'**avaluació intrínseca**, mesurant directament la qualitat dels embeddings mitjançant la correlació de Spearman entre les similituds predites i les humanes.

- **Spanish STS:** conjunt de parells de frases amb puntuacions de similitud semàntica. S'utilitza per a l'**avaluació extrínseca** (STS), mesurant la correlació de Pearson entre les similituds predites i les de referència.

- **Corpus Wikipedia en espanyol (`raw.es.tgz`):** corpus de text en cru utilitzat per entrenar els embeddings distribucionals.

## Mètriques d'avaluació

| Dataset | Mètrica |
|---|---|
| Multi-SimLex | Correlació de Spearman (ρ) |
| Spanish STS | Correlació de Pearson (r) |




## 0. Setup

In [1]:
#%pip install requests "datasets<3.0"

In [2]:
!pip install pandas scipy datasets tqdm

Defaulting to user installation because normal site-packages is not writeable
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   -- ------------------------------------- 1.8/27.4 MB 8.4 MB/s eta 0:00:04
   ---- ----------------------------------- 3.1/27.4 MB 8.0 MB/s eta 0:00:04
   ------- -------------------------------- 5.0/27.4 MB 7.9 MB/s eta 0:00:03
   --------- ------------------------------ 6.3/27.4 MB 7.9 MB/s eta 0:00:03
   ----------- ---------------------------- 7.9/27.4 MB 8.0 M

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
alibi 0.9.6 requires dill<0.4.0,>=0.3.0, but you have dill 0.4.1 which is incompatible.


In [3]:
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from pathlib import Path
from datasets import load_dataset
from tqdm.notebook import tqdm
import os
import nltk
import re
import numpy as np
from itertools import product
from gensim.models import Word2Vec
from scipy.stats import spearmanr
from sklearn.model_selection import KFold

d:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Datasets


Aquesta pràctica utilitza tres datasets:

| Dataset | Descripció | Ús |
|---|---|---|
| **Multi-SimLex** | Parells de paraules amb puntuacions de similitud semàntica | Avaluació intrínseca |
| **Spanish STS** | Parells de frases amb puntuacions de similitud semàntica | Avaluació extrínseca (STS) |
| **raw_es** | Corpus de la Wikipedia en espanyol | Entrenament d'embeddings |

A partir de `raw_es`, entrenarem i compararem models de **Word2Vec** i **fastText**.

In [4]:


os.system("wget -O SPA.csv https://web.archive.org/web/20231020014354/https://multisimlex.com/data/SPA.csv")

spa = pd.read_csv("SPA.csv")

annotator_cols = [c for c in spa.columns if c.startswith('Annotator')]
spa['SPA_AVG'] = spa[annotator_cols].mean(axis=1)

multi_simlex = spa[['ID', 'Word 1', 'Word 2', 'SPA_AVG']].copy()
multi_simlex.columns = ['PAIR_ID', 'SPA_W1', 'SPA_W2', 'SPA_AVG']

print(f"Multi-SimLex (ES) — {len(multi_simlex)} parells de paraules")
multi_simlex.head()

Multi-SimLex (ES) — 1888 parells de paraules


,PAIR_ID,SPA_W1,SPA_W2,SPA_AVG
0,1,brazo,músculo,1.4
1,2,democracia,monarquía,1.3
2,3,tejado,techo,4.8
3,4,amigo,profesor,0.4
4,5,mano,pie,1.1


In [5]:
# ============================================================
# DATASET 2 — Spanish STS (PlanTL-GOB-ES/sts-es)
# Avaluació extrínseca · Mètrica: correlació de Pearson
# ============================================================
sts = load_dataset("PlanTL-GOB-ES/sts-es")

train_df = sts["train"].to_pandas().rename(columns={"label": "score"})
dev_df   = sts["validation"].to_pandas().rename(columns={"label": "score"})
test_df  = sts["test"].to_pandas().rename(columns={"label": "score"})

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print(train_df.head())

Train: 1320 | Dev: 77 | Test: 155
  id                                          sentence1  \
0  0  Según el sondeo, 87% de los católicos cree que...   
1  1  La psicología explora conceptos como la percep...   
2  2  La tradición comenzó en el siglo IV, pero la m...   
3  3  "Maria Anna Schwegelin" (? - 1781 en la cárcel...   
4  4  Su identidad la había revelado durante el viaj...   

                                           sentence2  score  
0  El 87% de los católicos del mundo aprobaron el...   3.75  
1  La "psicología básica" es la parte de la psico...   2.80  
2  La tradición de erigir piedras con inscripcion...   2.40  
3  Te entregamos, Anna Schwegelin, al verdugo par...   2.20  
4  La información fue suministrada por el pescado...   2.20  


In [6]:
# ============================================================
# DATASET 3 — Corpus Wikipedia en espanyol
# Carpeta raw_es/ ja descomprimida, fitxers sense extensió o textos en pla
# ============================================================
corpus_dir   = "raw_es"
corpus_files = sorted(os.listdir(corpus_dir))

print(f"Total fitxers trobats: {len(corpus_files)}")
print(corpus_files[:5])

Total fitxers trobats: 57
['spanishText_10000_15000', 'spanishText_110000_115000', 'spanishText_120000_125000', 'spanishText_15000_20000', 'spanishText_180000_185000']


## 2. Preprocessament de *raw_es*

Primer de tot, cal realitzar un preprocessament en els corpus de *raw_es* per eliminar elements que no ens serveixen: </doc ...>, ENDOFARTICLE i URLs. També realitzarem altres preprocessaments més bàsics: eliminar *stopwords*, convertir a minúscules, eliminar espais múltiples i tokenitzar.



In [7]:
nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS    = set(stopwords.words('spanish'))

# funció per preprocessar cada línia del corpus
def preprocessa_linia(linia):
    # Saltar línies buides, etiquetes <doc ...> i ENDOFARTICLE
    linia = linia.strip()
    if not linia or linia.startswith("<") or linia == "ENDOFARTICLE.":
        return []

    # Minúscules
    linia = linia.lower()
    # Eliminar URLs
    linia = re.sub(r'https?://\S+|www\.\S+', '', linia)
    # Conservar només lletres (inclou accents, ñ, ü) i espais
    linia = re.sub(r'[^a-záéíóúüñà\s]', ' ', linia)
    # Eliminar espais múltiples
    linia = re.sub(r'\s+', ' ', linia).strip()

    # Tokenitzar, filtrar stopwords
    paraules = [p for p in linia.split() if p not in STOPWORDS]
    return paraules

def carrega_corpus(directori, fitxers):
    for nom_fitxer in fitxers:
        ruta = os.path.join(directori, nom_fitxer)
        with open(ruta, "r", encoding="latin-1") as f:
            for linia in f:
                paraules = preprocessa_linia(linia)
                if paraules:
                    yield paraules

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lihao\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
corpus = carrega_corpus(corpus_dir, corpus_files)

print("Primeres 3 frases preprocessades:")
for i, frase in enumerate(corpus):
    print(f"  [{i+1}] {frase[:10]}")
    if i == 2:
        break

Primeres 3 frases preprocessades:
  [1] ['acontecimientos']
  [2] ['nacimientos']
  [3] ['fallecimientos']


## 3. Models Word2Vec amb raw_es

El model Word2Vec és auto-supervisat, per tant no té un conjunt de validació amb etiquetes. El GridSearch requereix aquestes etiquetes per a la seva funció .fit(x.y), per tant no se li podrà aplicar. Com a solució,

In [9]:
class WikiCorpus:
    def __init__(self, directori, fitxers):
        self.directori = directori
        self.fitxers   = fitxers

    def __iter__(self):
        for nom_fitxer in self.fitxers:
            ruta = os.path.join(self.directori, nom_fitxer)
            with open(ruta, "r", encoding="latin-1") as f:
                for linia in f:
                    paraules = preprocessa_linia(linia)
                    if paraules:
                        yield paraules

### Word2Vec: CBOW

L'arquitectura *Continuous Bag of Words* de Word2Vec agafa el context i prediu la paraula central. En aquesta arquitectura, l'ordre de les paraules no importa, només quines paraules hi ha. Un exemple clàssic és quan volem completar una frase que li falta una paraula segons el context que ens dóna les altres paraules:
 - "El gat seu sobre la ___"

In [10]:
def avalua_model(model, parells_df, n_splits=5):
    """
    Calcula la correlació de Spearman mitjançant K-Fold Cross Validation
    sobre els parells de Multi-SimLex.
    Retorna la mitjana i desviació estàndard de les correlacions.
    """
    # Filtrar parells on ambdues paraules estan al vocabulari
    mascara = parells_df.apply(
        lambda r: r['SPA_W1'] in model.wv and r['SPA_W2'] in model.wv,
        axis=1
    )
    df_valid = parells_df[mascara].reset_index(drop=True)

    if len(df_valid) < n_splits:
        return np.nan, np.nan

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    correlacions = []

    for _, idx_test in kf.split(df_valid):
        fold = df_valid.iloc[idx_test]

        similituds_model = [
            model.wv.similarity(r['SPA_W1'], r['SPA_W2'])
            for _, r in fold.iterrows()
        ]
        similituds_gold = fold['SPA_AVG'].tolist()

        corr, _ = spearmanr(similituds_model, similituds_gold)
        correlacions.append(corr)

    return np.mean(correlacions), np.std(correlacions)

In [ ]:
    # ============================================================
    # GRID SEARCH RÀPID — Word2Vec CBOW
    # Estratègia: cercar hiperparàmetres sobre una mostra del corpus
    #             i ampliar el millor model amb més dades
    # ============================================================

# Si el model ja existeix, carreguem directament sense reentrenar
if os.path.exists("w2v_cbow_optimitzat.model"):
    print("Model ja existent, carregant w2v_cbow_optimitzat.model...")
    w2v_cbow = Word2Vec.load("w2v_cbow_optimitzat.model")
    print(f"Model carregat ✓ — vocabulari: {len(w2v_cbow.wv):,} paraules")

else:
    print("Model no trobat, iniciant entrenament...")´
    
    # ── FASE 1: CARREGAR MOSTRA A RAM ────────────────────────────────────────────
    N_FITXERS_MOSTRA = 5

    print(f"Carregant mostra ({N_FITXERS_MOSTRA} fitxers) a memòria...")
    corpus_mostra = list(WikiCorpus(corpus_dir, corpus_files[:N_FITXERS_MOSTRA]))
    print(f"Frases a la mostra: {len(corpus_mostra):,}")

    # ── FASE 2: GRID SEARCH SOBRE LA MOSTRA ─────────────────────────────────────
    param_grid = {
        'vector_size': [100, 300],
        'window':      [3, 5],
        'min_count':   [5, 10],
        'epochs':      [5, 10],
    }

    claus        = list(param_grid.keys())
    valors       = list(param_grid.values())
    combinacions = list(product(*valors))

    print(f"\nTotal combinacions: {len(combinacions)}\n")

    resultats    = []
    millor_model = None       # ← guardem el millor model durant el bucle
    millor_score = -np.inf

    for i, combo in enumerate(combinacions):
        params = dict(zip(claus, combo))
        print(f"[{i+1}/{len(combinacions)}] Entrenant amb {params}...")

        model = Word2Vec(
            sentences=corpus_mostra,
            vector_size=params['vector_size'],
            window=params['window'],
            min_count=params['min_count'],
            sg=0,
            workers=4,
            epochs=params['epochs']
        )

        mitjana, std = avalua_model(model, multi_simlex, n_splits=5)
        resultats.append({**params, 'spearman_mean': mitjana, 'spearman_std': std})
        print(f"  → Spearman: {mitjana:.4f} ± {std:.4f}")

        # Guardem el model si és el millor fins ara
        if mitjana > millor_score:
            millor_score = mitjana
            millor_model = model

    # ── RESULTATS ────────────────────────────────────────────────────────────────
    resultats_df = pd.DataFrame(resultats).sort_values('spearman_mean', ascending=False)
    print("\nRànquing de models:")
    print(resultats_df.to_string(index=False))

    millors = resultats_df.iloc[0]
    print(f"\nMillors hiperparàmetres:")
    print(f"  vector_size : {int(millors['vector_size'])}")
    print(f"  window      : {int(millors['window'])}")
    print(f"  min_count   : {int(millors['min_count'])}")
    print(f"  epochs      : {int(millors['epochs'])}")
    print(f"  Spearman    : {millors['spearman_mean']:.4f} ± {millors['spearman_std']:.4f}")

    # ── FASE 3: AMPLIAR EL MILLOR MODEL AMB MÉS DADES ───────────────────────────
    N_FITXERS_FINAL = 20

    print(f"\nAmpliant el millor model amb {N_FITXERS_FINAL} fitxers...")
    corpus_final = list(WikiCorpus(corpus_dir, corpus_files[:N_FITXERS_FINAL]))

    # Ampliar el vocabulari amb les noves dades
    millor_model.build_vocab(corpus_final, update=True)

    # Continuar l'entrenament sobre les noves dades
    millor_model.train(
        corpus_final,
        total_examples=len(corpus_final),
        epochs=millor_model.epochs
    )

    w2v_cbow = millor_model
    w2v_cbow.save("w2v_cbow_optimitzat.model")
    print("Model CBOW optimitzat guardat ✓")

    # tarda 10 min casi

Carregant mostra (5 fitxers) a memòria...
Frases a la mostra: 531,869

Total combinacions: 16

[1/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 5}...
  → Spearman: 0.2619 ± 0.0278
[2/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 10}...
  → Spearman: 0.3087 ± 0.0378
[3/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 5}...
  → Spearman: 0.2631 ± 0.0267
[4/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 10}...
  → Spearman: 0.3119 ± 0.0270
[5/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5}...
  → Spearman: 0.2755 ± 0.0374
[6/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 10}...
  → Spearman: 0.3242 ± 0.0436
[7/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 10, 'epochs': 5}...
  → Spearman: 0.2851 ± 0.0284
[8/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 10, 'epochs'

Importem el model òptim guardat.

In [14]:
from gensim.models import Word2Vec

w2v_cbow = Word2Vec.load("w2v_cbow_optimitzat.model")

print(f"Vocabulari: {len(w2v_cbow.wv):,} paraules")
print(f"Dimensió dels vectors: {w2v_cbow.vector_size}")

Vocabulari: 172,689 paraules
Dimensió dels vectors: 100


### Word2Vec: Skip-gram

L'estructura *Skip-gram* de Word2Vec fa el procés invers a CBOW: ara sabem la paraula central, així que volem predir les paraules que estiguin al voltant (el context). Aquest model prediu per a cada posició de la finestra de manera independent, per tant pot generar molts exemples possibles, i això fa que el seu entrenament sigui més lent que CBOW. L'avantatge és que aprèn millor les paraules menys freqüents: com que genera molts exemples per cada paraula, fins i tot les poc freqüents es veuen prou vegades per aprendre una bona representació.

## 4. Models FastText